# ERP Enrichment Starter: Maintenance Event Triage

This notebook demonstrates the ODS-E ERP enrichment schemas end-to-end — from loading schemas and validating synthetic data to walking through a complete maintenance event triage workflow with visualizations.

**What you'll do:**
1. Load all 7 ERP enrichment JSON schemas
2. Create synthetic maintenance data for a fictional solar farm (Kariega Solar Farm)
3. Validate every record against its schema
4. Walk through a SCADA alarm triage scenario using 5 schemas
5. Visualize alarm escalation, equipment hierarchy, and spare parts inventory

**Self-contained:** The first cell installs all dependencies. No prior setup required.

**Schemas used:**
- `equipment-id-map.json` — canonical ID to native system ID mapping
- `equipment-register.json` — equipment hierarchy and technical attributes
- `maintenance-history.json` — work order records
- `spare-parts.json` — inventory levels and reorder thresholds
- `procurement-context.json` — supplier and purchase order context
- `failure-taxonomy.json` — failure codes and repair statistics
- `alarm-frequency-profile.json` — derived alarm analytics from SCADA logs

In [ ]:
# Install dependencies (safe to re-run — pip skips already-installed packages)
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "matplotlib", "jsonschema"
])
print("All dependencies installed.")

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from jsonschema import validate, ValidationError

# Style — matches ona-protocol notebook conventions
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#cccccc',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#cccccc',
})

BLUE = '#455BF1'
RED = '#E20419'
DARK = '#1a1a2e'
GREY = '#6b7280'
AMBER = '#f59e0b'
GREEN = '#10b981'

# Path setup — schemas are in the repo root
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
SCHEMA_DIR = os.path.join(NOTEBOOK_DIR, '..', 'schemas')

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Schema directory:   {os.path.abspath(SCHEMA_DIR)}")

---
## Step 1: Load the ERP Enrichment Schemas

ODS-E defines 7 JSON schemas for ERP enrichment. Each schema specifies required and optional fields, types, enums, and constraints. We load them all and print a summary.

In [ ]:
SCHEMA_FILES = [
    'equipment-id-map.json',
    'equipment-register.json',
    'maintenance-history.json',
    'spare-parts.json',
    'procurement-context.json',
    'failure-taxonomy.json',
    'alarm-frequency-profile.json',
]

schemas = {}
for fname in SCHEMA_FILES:
    path = os.path.join(SCHEMA_DIR, fname)
    with open(path) as f:
        schemas[fname] = json.load(f)

print(f"Loaded {len(schemas)} schemas:\n")
for fname, schema in schemas.items():
    title = schema.get('title', fname)
    schema_type = schema.get('type', '?')
    # For array types, required is on items; for object types, on root
    if schema_type == 'array':
        item_props = schema.get('items', {}).get('properties', {})
        required = schema.get('items', {}).get('required', [])
        total_fields = len(item_props)
    else:
        item_props = schema.get('properties', {})
        required = schema.get('required', [])
        total_fields = len(item_props)
    print(f"  {title}")
    print(f"    Type: {schema_type} | Required: {len(required)} | Total fields: {total_fields}")
    print(f"    Required: {', '.join(required)}")
    print()

---
## Step 2: Create Synthetic Data for Kariega Solar Farm

**Kariega Solar Farm** is a fictional 10 MW ground-mount solar installation in Eastern Cape, South Africa. It runs SMA string inverters monitored through a site SCADA system, with IFS Cloud as the ERP.

We create synthetic records for all 7 schemas — these represent what an IFS Cloud transform and alarm frequency computation would produce.

In [ ]:
# --- Equipment ID Map (array of 6 equipment entries) ---
equipment_id_map = [
    {
        "equipment_id": "KRG-SITE",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10000"},
            {"system": "site-scada", "type": "scada", "native_id": "KARIEGA-MAIN"}
        ]
    },
    {
        "equipment_id": "KRG-ARR-A",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10010"},
            {"system": "site-scada", "type": "scada", "native_id": "ARR-A-SCADA"}
        ]
    },
    {
        "equipment_id": "KRG-INV-001",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10045"},
            {"system": "sma-monitoring", "type": "monitoring", "native_id": "SMA-9801234"},
            {"system": "site-scada", "type": "scada", "native_id": "INV-001-SCADA"}
        ]
    },
    {
        "equipment_id": "KRG-INV-002",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10046"},
            {"system": "sma-monitoring", "type": "monitoring", "native_id": "SMA-9801235"},
            {"system": "site-scada", "type": "scada", "native_id": "INV-002-SCADA"}
        ]
    },
    {
        "equipment_id": "KRG-INV-003",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10047"},
            {"system": "sma-monitoring", "type": "monitoring", "native_id": "SMA-9801236"},
            {"system": "site-scada", "type": "scada", "native_id": "INV-003-SCADA"}
        ]
    },
    {
        "equipment_id": "KRG-TRF-001",
        "sources": [
            {"system": "ifs-cloud", "type": "erp", "native_id": "EQ-10080"},
            {"system": "site-scada", "type": "scada", "native_id": "TRF-001-SCADA"}
        ]
    }
]

# --- Equipment Register (8 records) ---
equipment_register = [
    {"equipment_id": "KRG-SITE", "site_id": "KARIEGA-SOLAR", "equipment_type": "site",
     "design_capacity_kw": 10000.0, "install_date": "2023-03-01"},
    {"equipment_id": "KRG-ARR-A", "site_id": "KARIEGA-SOLAR", "equipment_type": "array",
     "parent_equipment_id": "KRG-SITE", "design_capacity_kw": 5000.0},
    {"equipment_id": "KRG-ARR-B", "site_id": "KARIEGA-SOLAR", "equipment_type": "array",
     "parent_equipment_id": "KRG-SITE", "design_capacity_kw": 5000.0},
    {"equipment_id": "KRG-INV-001", "site_id": "KARIEGA-SOLAR", "equipment_type": "inverter",
     "parent_equipment_id": "KRG-ARR-A", "manufacturer": "SMA",
     "model": "Sunny Tripower 25000TL", "serial_number": "SMA-9801234",
     "install_date": "2023-06-15", "warranty_expiry": "2028-06-15",
     "design_capacity_kw": 25.0, "cost_center": "CC-KRG-SOLAR"},
    {"equipment_id": "KRG-INV-002", "site_id": "KARIEGA-SOLAR", "equipment_type": "inverter",
     "parent_equipment_id": "KRG-ARR-A", "manufacturer": "SMA",
     "model": "Sunny Tripower 25000TL", "serial_number": "SMA-9801235",
     "install_date": "2023-06-15", "warranty_expiry": "2028-06-15",
     "design_capacity_kw": 25.0, "cost_center": "CC-KRG-SOLAR"},
    {"equipment_id": "KRG-INV-003", "site_id": "KARIEGA-SOLAR", "equipment_type": "inverter",
     "parent_equipment_id": "KRG-ARR-B", "manufacturer": "SMA",
     "model": "Sunny Tripower 25000TL", "serial_number": "SMA-9801236",
     "install_date": "2023-06-15", "warranty_expiry": "2028-06-15",
     "design_capacity_kw": 25.0, "cost_center": "CC-KRG-SOLAR"},
    {"equipment_id": "KRG-TRF-001", "site_id": "KARIEGA-SOLAR", "equipment_type": "transformer",
     "parent_equipment_id": "KRG-SITE", "manufacturer": "ABB",
     "model": "DTR-11kV-2500kVA", "design_capacity_kw": 2500.0},
    {"equipment_id": "KRG-MTR-001", "site_id": "KARIEGA-SOLAR", "equipment_type": "meter",
     "parent_equipment_id": "KRG-TRF-001", "manufacturer": "Itron",
     "model": "ACE6000"}
]

# --- Maintenance History (4 work orders) ---
maintenance_history = [
    {"equipment_id": "KRG-INV-002", "work_order_id": "WO-2025-0891",
     "wo_type": "corrective", "wo_status": "closed",
     "reported_date": "2025-11-02T09:15:00Z", "completed_date": "2025-11-02T15:30:00Z",
     "downtime_hours": 6.25, "failure_code": "GRID_FAULT", "cause_code": "FUSE_BLOWN",
     "total_cost": 1250.00, "parts_consumed": [{"part_id": "SP-FUSE-60A", "qty": 2}]},
    {"equipment_id": "KRG-INV-002", "work_order_id": "WO-2026-0142",
     "wo_type": "corrective", "wo_status": "closed",
     "reported_date": "2026-01-15T08:30:00Z", "completed_date": "2026-01-15T14:45:00Z",
     "downtime_hours": 6.0, "failure_code": "GRID_FAULT", "cause_code": "FUSE_BLOWN",
     "total_cost": 1180.00, "parts_consumed": [{"part_id": "SP-FUSE-60A", "qty": 2}]},
    {"equipment_id": "KRG-INV-001", "work_order_id": "WO-2026-0098",
     "wo_type": "preventive", "wo_status": "closed",
     "reported_date": "2026-01-10T07:00:00Z", "completed_date": "2026-01-10T11:00:00Z",
     "downtime_hours": 4.0, "total_cost": 450.00},
    {"equipment_id": "KRG-TRF-001", "work_order_id": "WO-2026-0201",
     "wo_type": "condition_based", "wo_status": "open",
     "reported_date": "2026-02-20T10:00:00Z",
     "failure_code": "OIL_TEMP_HIGH", "cause_code": "COOLING_FAN_FAIL"}
]

# --- Spare Parts (4 parts) ---
spare_parts = [
    {"part_id": "SP-FUSE-60A", "description": "60A DC string fuse, 1000V",
     "equipment_types_served": ["inverter", "combiner"],
     "qty_on_hand": 24, "qty_reserved": 4, "qty_available": 20,
     "reorder_point": 10, "supplier_lead_time_days": 14, "last_unit_cost": 18.50},
    {"part_id": "SP-SPD-40KA", "description": "40kA surge protection device, Type 2",
     "equipment_types_served": ["inverter"],
     "qty_on_hand": 6, "qty_reserved": 0, "qty_available": 6,
     "reorder_point": 4, "supplier_lead_time_days": 21, "last_unit_cost": 285.00},
    {"part_id": "SP-FAN-COOL", "description": "Transformer cooling fan assembly",
     "equipment_types_served": ["transformer"],
     "qty_on_hand": 2, "qty_reserved": 1, "qty_available": 1,
     "reorder_point": 2, "supplier_lead_time_days": 35, "last_unit_cost": 1450.00},
    {"part_id": "SP-CONN-MC4", "description": "MC4 connector pair, 30A",
     "equipment_types_served": ["string", "combiner"],
     "qty_on_hand": 120, "qty_reserved": 0, "qty_available": 120,
     "reorder_point": 50, "supplier_lead_time_days": 7, "last_unit_cost": 3.20}
]

# --- Procurement Context (4 records) ---
procurement_context = [
    {"part_id": "SP-FUSE-60A", "preferred_supplier": "Phoenix Contact SA",
     "avg_lead_time_days": 12, "avg_unit_cost": 17.80,
     "last_po_date": "2026-01-20", "open_po_qty": 50, "open_po_eta": "2026-02-10"},
    {"part_id": "SP-SPD-40KA", "preferred_supplier": "Dehn Africa",
     "avg_lead_time_days": 18, "avg_unit_cost": 275.00,
     "last_po_date": "2025-12-05"},
    {"part_id": "SP-FAN-COOL", "preferred_supplier": "ABB South Africa",
     "avg_lead_time_days": 30, "avg_unit_cost": 1380.00,
     "last_po_date": "2025-10-15", "open_po_qty": 2, "open_po_eta": "2026-03-01"},
    {"part_id": "SP-CONN-MC4", "preferred_supplier": "Staubli Electrical SA",
     "avg_lead_time_days": 5, "avg_unit_cost": 2.95,
     "last_po_date": "2026-02-01"}
]

# --- Failure Taxonomy (4 failure codes) ---
failure_taxonomy = [
    {"failure_code": "GRID_FAULT",
     "failure_description": "Grid voltage or frequency excursion causing inverter trip",
     "cause_code": "FUSE_BLOWN",
     "cause_description": "DC string fuse failure due to thermal cycling",
     "recurrence_rate": 0.35, "typical_mttr_hours": 4.5, "typical_cost": 1200.00},
    {"failure_code": "OIL_TEMP_HIGH",
     "failure_description": "Transformer oil temperature exceeding threshold",
     "cause_code": "COOLING_FAN_FAIL",
     "cause_description": "Cooling fan motor bearing failure",
     "recurrence_rate": 0.15, "typical_mttr_hours": 8.0, "typical_cost": 3200.00},
    {"failure_code": "COMM_LOSS",
     "failure_description": "Communication loss between inverter and monitoring system",
     "cause_code": "CABLE_DAMAGE",
     "cause_description": "RS485 communication cable damaged by rodent or UV",
     "recurrence_rate": 0.10, "typical_mttr_hours": 2.0, "typical_cost": 350.00},
    {"failure_code": "ISOL_FAULT",
     "failure_description": "Ground insulation resistance below threshold",
     "cause_code": "CONNECTOR_DEGRADE",
     "cause_description": "MC4 connector degradation from moisture ingress",
     "recurrence_rate": 0.20, "typical_mttr_hours": 3.0, "typical_cost": 500.00}
]

# --- Alarm Frequency Profiles (5 profiles) ---
alarm_profiles = [
    {"equipment_id": "KRG-INV-002", "alarm_code": "GRID_FAULT_UV",
     "source_equipment_id": "INV-002-SCADA",
     "count_7d": 12, "count_30d": 18, "count_90d": 25,
     "mean_time_between_alarms_hours": 86.4, "escalation_rate": 2.86,
     "prior_wo_count_same_alarm": 2, "prior_resolution": "recurring"},
    {"equipment_id": "KRG-INV-001", "alarm_code": "GRID_FAULT_UV",
     "source_equipment_id": "INV-001-SCADA",
     "count_7d": 1, "count_30d": 4, "count_90d": 8,
     "mean_time_between_alarms_hours": 270.0, "escalation_rate": 1.07,
     "prior_wo_count_same_alarm": 0, "prior_resolution": "none"},
    {"equipment_id": "KRG-INV-003", "alarm_code": "COMM_LOSS",
     "source_equipment_id": "INV-003-SCADA",
     "count_7d": 3, "count_30d": 5, "count_90d": 7,
     "mean_time_between_alarms_hours": 308.6, "escalation_rate": 2.57,
     "prior_wo_count_same_alarm": 0, "prior_resolution": "none"},
    {"equipment_id": "KRG-TRF-001", "alarm_code": "OIL_TEMP_HIGH",
     "source_equipment_id": "TRF-001-SCADA",
     "count_7d": 2, "count_30d": 3, "count_90d": 3,
     "mean_time_between_alarms_hours": 720.0, "escalation_rate": 2.86,
     "prior_wo_count_same_alarm": 1, "prior_resolution": "closed_deferred"},
    {"equipment_id": "KRG-INV-002", "alarm_code": "ISOL_FAULT",
     "source_equipment_id": "INV-002-SCADA",
     "count_7d": 0, "count_30d": 2, "count_90d": 5,
     "mean_time_between_alarms_hours": 432.0, "escalation_rate": 0.0,
     "prior_wo_count_same_alarm": 0, "prior_resolution": "none"}
]

print(f"Equipment ID map:       {len(equipment_id_map)} entries")
print(f"Equipment register:     {len(equipment_register)} records")
print(f"Maintenance history:    {len(maintenance_history)} work orders")
print(f"Spare parts:            {len(spare_parts)} parts")
print(f"Procurement context:    {len(procurement_context)} records")
print(f"Failure taxonomy:       {len(failure_taxonomy)} failure codes")
print(f"Alarm frequency:        {len(alarm_profiles)} profiles")

---
## Step 3: Validate Against Schemas

Every synthetic record must pass JSON Schema validation. This is the same check production pipelines run after ERP extraction.

In [ ]:
def validate_records(records, schema, name):
    """Validate a list of records against a schema. Returns (pass_count, fail_count)."""
    passed, failed = 0, 0
    for i, record in enumerate(records):
        try:
            validate(instance=record, schema=schema)
            passed += 1
        except ValidationError as e:
            failed += 1
            print(f"  FAIL {name}[{i}]: {e.message}")
    return passed, failed

results = []

# Equipment ID Map — array schema, validate the whole array
try:
    validate(instance=equipment_id_map, schema=schemas['equipment-id-map.json'])
    results.append(('Equipment ID Map', len(equipment_id_map), 0))
except ValidationError as e:
    results.append(('Equipment ID Map', 0, len(equipment_id_map)))
    print(f"  FAIL equipment-id-map: {e.message}")

# Object-type schemas — validate each record individually
schema_record_pairs = [
    ('equipment-register.json', equipment_register, 'Equipment Register'),
    ('maintenance-history.json', maintenance_history, 'Maintenance History'),
    ('spare-parts.json', spare_parts, 'Spare Parts'),
    ('procurement-context.json', procurement_context, 'Procurement Context'),
    ('failure-taxonomy.json', failure_taxonomy, 'Failure Taxonomy'),
    ('alarm-frequency-profile.json', alarm_profiles, 'Alarm Frequency Profile'),
]

for schema_key, records, name in schema_record_pairs:
    p, f = validate_records(records, schemas[schema_key], name)
    results.append((name, p, f))

# Summary
print(f"\n{'Schema':<30} {'Passed':>8} {'Failed':>8}")
print(f"{'='*48}")
total_pass, total_fail = 0, 0
for name, p, f in results:
    status = 'PASS' if f == 0 else 'FAIL'
    print(f"{name:<30} {p:>8} {f:>8}  {status}")
    total_pass += p
    total_fail += f
print(f"{'='*48}")
print(f"{'TOTAL':<30} {total_pass:>8} {total_fail:>8}")
print(f"\nAll records valid!" if total_fail == 0 else f"\n{total_fail} record(s) failed validation.")

---
## Step 4: Maintenance Event Triage

It's 08:30 at Kariega Solar Farm. Inverter `KRG-INV-002` just triggered a `GRID_FAULT_UV` alarm (grid undervoltage fault). The O&M control room needs to decide: dispatch now, schedule, or monitor?

We walk through the 4-step triage workflow using the ERP enrichment data.

In [ ]:
# Step 4a: Alarm Frequency Check
target_equip = "KRG-INV-002"
target_alarm = "GRID_FAULT_UV"

alarm = next(p for p in alarm_profiles
             if p["equipment_id"] == target_equip and p["alarm_code"] == target_alarm)

escalating = alarm["escalation_rate"] > 1.0

print(f"STEP 1: Alarm Frequency Check")
print(f"{'='*45}")
print(f"Equipment:        {alarm['equipment_id']}")
print(f"Alarm:            {alarm['alarm_code']}")
print(f"Count (7d):       {alarm['count_7d']}")
print(f"Count (30d):      {alarm['count_30d']}")
print(f"Count (90d):      {alarm['count_90d']}")
print(f"Escalation rate:  {alarm['escalation_rate']:.2f}")
print(f"MTBA:             {alarm['mean_time_between_alarms_hours']:.1f} hours")
print(f"Prior WOs:        {alarm['prior_wo_count_same_alarm']}")
print(f"Prior resolution: {alarm['prior_resolution']}")
print(f"\nEscalating? {'YES — alarm rate is {:.1f}x the 30-day baseline'.format(alarm['escalation_rate']) if escalating else 'No'}")

In [ ]:
# Step 4b: Maintenance History + Failure Taxonomy
equip_wos = [wo for wo in maintenance_history if wo["equipment_id"] == target_equip]

# Find the relevant failure code from prior WOs
failure_codes_seen = set(wo.get("failure_code") for wo in equip_wos if wo.get("failure_code"))
failure_info = next((ft for ft in failure_taxonomy if ft["failure_code"] in failure_codes_seen), None)

recurring = failure_info["recurrence_rate"] > 0.2 if failure_info else False

print(f"STEP 2: Maintenance History + Failure Taxonomy")
print(f"{'='*50}")
print(f"Prior work orders on {target_equip}: {len(equip_wos)}")
for wo in equip_wos:
    print(f"  {wo['work_order_id']}  {wo['wo_type']:<15} {wo['wo_status']:<12} "
          f"{wo.get('failure_code', '-'):<15} R {wo.get('total_cost', 0):>8,.2f}")

if failure_info:
    print(f"\nFailure taxonomy:")
    print(f"  Code:            {failure_info['failure_code']}")
    print(f"  Description:     {failure_info['failure_description']}")
    print(f"  Root cause:      {failure_info['cause_description']}")
    print(f"  Recurrence rate: {failure_info['recurrence_rate']:.0%}")
    print(f"  Typical MTTR:    {failure_info['typical_mttr_hours']} hours")
    print(f"  Typical cost:    R {failure_info['typical_cost']:,.2f}")

print(f"\nRecurring? {'YES — {:.0%} recurrence rate'.format(failure_info['recurrence_rate']) if recurring else 'No'}")

In [ ]:
# Step 4c: Spare Parts + Procurement
# Find parts consumed in prior WOs for this failure
parts_needed = {}
for wo in equip_wos:
    for part in wo.get("parts_consumed", []):
        parts_needed[part["part_id"]] = parts_needed.get(part["part_id"], 0) + part["qty"]

# Average qty per WO
if equip_wos:
    for pid in parts_needed:
        parts_needed[pid] = parts_needed[pid] / len([wo for wo in equip_wos if wo.get("parts_consumed")])

print(f"STEP 3: Spare Parts + Procurement")
print(f"{'='*55}")

all_parts_available = True
for pid, avg_qty in parts_needed.items():
    needed = int(np.ceil(avg_qty))
    sp = next((s for s in spare_parts if s["part_id"] == pid), None)
    pc = next((p for p in procurement_context if p["part_id"] == pid), None)
    if sp:
        available = sp["qty_available"] >= needed
        if not available:
            all_parts_available = False
        print(f"Part: {sp['description']} ({pid})")
        print(f"  On hand:     {sp['qty_on_hand']}")
        print(f"  Reserved:    {sp['qty_reserved']}")
        print(f"  Available:   {sp['qty_available']}")
        print(f"  Needed:      {needed}")
        print(f"  In stock:    {'YES' if available else 'NO'}")
        if pc and pc.get('open_po_qty'):
            print(f"  Open PO:     {pc['open_po_qty']} units, ETA {pc.get('open_po_eta', 'TBD')}")
        print()

print(f"All parts available? {'YES' if all_parts_available else 'NO — supply chain blocker'}")

In [ ]:
# Step 4d: Decision Matrix
if escalating and recurring:
    if all_parts_available:
        action = "DISPATCH NOW"
        reason = "Escalating alarm with recurring failure pattern — parts in stock"
        priority = "CRITICAL"
    else:
        action = "ORDER PARTS + SCHEDULE"
        reason = "Escalating alarm with recurring failure — waiting on parts"
        priority = "HIGH"
elif escalating:
    action = "SCHEDULE INSPECTION"
    reason = "Escalating alarm but no prior failure history — investigate first"
    priority = "MEDIUM"
elif recurring:
    action = "SCHEDULE MAINTENANCE"
    reason = "Known recurring failure but alarm rate is stable"
    priority = "MEDIUM"
else:
    action = "MONITOR"
    reason = "Low alarm rate, no recurring failure pattern"
    priority = "LOW"

print(f"\n{'='*55}")
print(f"  TRIAGE DECISION")
print(f"{'='*55}")
print(f"  Action:     {action}")
print(f"  Priority:   {priority}")
print(f"  Reason:     {reason}")
print(f"{'='*55}")
print(f"\n  Equipment:    {alarm['equipment_id']}")
print(f"  Alarm:        {alarm['alarm_code']}")
print(f"  Escalating:   {'Yes' if escalating else 'No'} (rate: {alarm['escalation_rate']:.2f})")
print(f"  Recurring:    {'Yes' if recurring else 'No'}", end='')
if failure_info:
    print(f" (rate: {failure_info['recurrence_rate']:.0%})")
else:
    print()
print(f"  Parts ready:  {'Yes' if all_parts_available else 'No'}")
if failure_info:
    print(f"  Est. MTTR:    {failure_info['typical_mttr_hours']} hours")
    print(f"  Est. cost:    R {failure_info['typical_cost']:,.2f}")

---
## Step 5: Visualizations

Three views into the ERP enrichment data: alarm escalation trends, equipment hierarchy, and spare parts inventory status.

In [ ]:
# Visualization 1: Alarm Escalation — Grouped Bar Chart
# Show count_7d, count_30d, count_90d per alarm profile
labels = [f"{p['equipment_id']}\n{p['alarm_code']}" for p in alarm_profiles]
count_7d = [p['count_7d'] for p in alarm_profiles]
count_30d = [p['count_30d'] for p in alarm_profiles]
count_90d = [p['count_90d'] for p in alarm_profiles]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars_7d = ax.bar(x - width, count_7d, width, label='7-day', color=RED, alpha=0.9)
bars_30d = ax.bar(x, count_30d, width, label='30-day', color=AMBER, alpha=0.9)
bars_90d = ax.bar(x + width, count_90d, width, label='90-day', color=BLUE, alpha=0.9)

ax.set_ylabel('Alarm Count')
ax.set_title('Alarm Frequency by Equipment and Window\n'
             'Kariega Solar Farm — Trailing 7 / 30 / 90 Day Windows')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend(fontsize=10)

# Annotate escalation rates
for i, p in enumerate(alarm_profiles):
    rate = p['escalation_rate']
    color = RED if rate > 1.5 else (AMBER if rate > 1.0 else GREEN)
    ax.text(x[i], max(count_90d[i], count_30d[i], count_7d[i]) + 1,
            f'esc: {rate:.1f}x', ha='center', fontsize=9, fontweight='bold', color=color)

ax.set_ylim(0, max(count_90d) * 1.25)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Equipment Hierarchy Tree (Unicode box-drawing)
# Build from equipment register using parent_equipment_id

def build_tree(records):
    """Build a dict of parent -> children from equipment register."""
    children = {}
    by_id = {r['equipment_id']: r for r in records}
    roots = []
    for r in records:
        pid = r.get('parent_equipment_id')
        if pid and pid in by_id:
            children.setdefault(pid, []).append(r['equipment_id'])
        elif not pid:
            roots.append(r['equipment_id'])
    return roots, children, by_id

def print_tree(eid, children, by_id, prefix='', is_last=True, lines=None):
    """Render tree using Unicode box-drawing characters."""
    if lines is None:
        lines = []
    r = by_id[eid]
    connector = '\u2514\u2500\u2500 ' if is_last else '\u251c\u2500\u2500 '
    etype = r['equipment_type'].upper()
    cap = f" ({r['design_capacity_kw']:.0f} kW)" if r.get('design_capacity_kw') else ''
    mfr = f" [{r.get('manufacturer', '')} {r.get('model', '')}]" if r.get('manufacturer') else ''
    line = f"{prefix}{connector}{eid}  {etype}{cap}{mfr}"
    lines.append(line)

    child_prefix = prefix + ('    ' if is_last else '\u2502   ')
    kids = children.get(eid, [])
    for i, kid in enumerate(kids):
        print_tree(kid, children, by_id, child_prefix, i == len(kids) - 1, lines)
    return lines

roots, children_map, by_id = build_tree(equipment_register)

print(f"Equipment Hierarchy — Kariega Solar Farm")
print(f"{'='*60}\n")

for root in roots:
    r = by_id[root]
    cap = f" ({r['design_capacity_kw']:.0f} kW)" if r.get('design_capacity_kw') else ''
    print(f"{root}  {r['equipment_type'].upper()}{cap}")
    kids = children_map.get(root, [])
    for i, kid in enumerate(kids):
        tree_lines = print_tree(kid, children_map, by_id, '', i == len(kids) - 1)
        for line in tree_lines:
            print(line)

In [ ]:
# Visualization 3: Spare Parts Stock vs Reorder Point — Horizontal Bar Chart
part_labels = [f"{sp['part_id']}\n{sp['description'][:30]}" for sp in spare_parts]
qty_on_hand = [sp['qty_on_hand'] for sp in spare_parts]
qty_reserved = [sp['qty_reserved'] for sp in spare_parts]
qty_available = [sp['qty_available'] for sp in spare_parts]
reorder_pts = [sp['reorder_point'] for sp in spare_parts]

y = np.arange(len(part_labels))
height = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

# Available stock (green if above reorder, amber/red if at or below)
bar_colors = [GREEN if qa > rp else (AMBER if qa > 0 else RED)
              for qa, rp in zip(qty_available, reorder_pts)]
ax.barh(y - height/2, qty_available, height, label='Available', color=bar_colors, alpha=0.9)
ax.barh(y + height/2, qty_reserved, height, label='Reserved', color=GREY, alpha=0.6)

# Reorder point markers
for i, rp in enumerate(reorder_pts):
    ax.plot(rp, i, 'D', color=RED, markersize=8, zorder=5)
ax.plot([], [], 'D', color=RED, markersize=8, label='Reorder Point')

ax.set_yticks(y)
ax.set_yticklabels(part_labels, fontsize=9)
ax.set_xlabel('Quantity')
ax.set_title('Spare Parts Inventory Status\n'
             'Kariega Solar Farm — Available Stock vs Reorder Point')
ax.legend(loc='lower right', fontsize=10)
ax.invert_yaxis()

# Value labels
for i in range(len(spare_parts)):
    if qty_available[i] > 0:
        ax.text(qty_available[i] + 1, i - height/2,
                str(qty_available[i]), va='center', fontsize=9, color=DARK)
    if qty_reserved[i] > 0:
        ax.text(qty_reserved[i] + 1, i + height/2,
                str(qty_reserved[i]), va='center', fontsize=9, color=GREY)

plt.tight_layout()
plt.show()

---
## Next Steps for Your Team

This notebook uses synthetic data to demonstrate the pattern. To adapt it for your sites:

1. **Connect to real data** — Replace the inline dictionaries with actual IFS Cloud extractions using the [`erp-transforms/ifs-cloud.yaml`](https://github.com/AsobaCloud/ona-protocol/blob/main/erp-transforms/ifs-cloud.yaml) transform specification. Start with equipment register and maintenance history.

2. **Build your alarm-to-failure mapping** — The triage workflow assumes a 1:1 mapping between SCADA alarm codes and ERP failure codes. In production, create a site-specific mapping table. See the [alarm frequency computation spec](https://github.com/AsobaCloud/ona-protocol/blob/main/spec/alarm-frequency-computation.md) for details.

3. **Tune decision thresholds** — The decision matrix uses `escalation_rate > 1.0` and `recurrence_rate > 0.2` as defaults. Calibrate these for your portfolio's risk tolerance and dispatch capacity.

4. **Add automated alerts** — Wrap the triage logic in a scheduled job that runs every 4 hours (matching ERP refresh cadence) and sends alerts for DISPATCH NOW and ORDER PARTS decisions.

5. **Build a dashboard** — Use the grouped bar chart and inventory status patterns as starting points for a live Grafana or Power BI dashboard fed by your ERP extraction pipeline.

**Documentation:**
- [ERP Enrichment Schemas](https://opendataschema.energy/docs/schemas/erp-enrichment) — Full field reference
- [Maintenance Event Triage Pattern](https://opendataschema.energy/docs/patterns/maintenance-event-triage) — Detailed walkthrough
- [Schema Overview](https://opendataschema.energy/docs/schemas/overview) — All ODS-E schema surfaces
- [ODS-E on GitHub](https://github.com/AsobaCloud/odse) — Schema definitions, transforms, and validation tools